In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torch.utils.data import ConcatDataset, DataLoader

import sys
import os

current_dir = os.path.dirname(os.path.abspath(__file__))
sys.path.append(current_dir)

from utils.EEDDataLoader import EEGDataset
from utils.train_eval import eval, train
from models.ConvParallelEEG1DModel import ConvParallelEEG1DModel

files_list_csv = "./dataset/meta_data.csv"
train_data_csv = "./dataset/train.csv"
val_data_csv = "./dataset/eval.csv"
test_data_csv = "./dataset/test.csv"
prediction_saving_csv_train = "./dataset/predictions/predictions_train_conv_crops.csv"
prediction_saving_csv_eval = "./dataset/predictions/predictions_eval_conv_crops.csv"
prediction_saving_csv_test = "./dataset/predictions/predictions_test_conv_crops.csv"
model_saving_name = "./dataset/model/weights/conv_tt_0_original.pt"

epochs = 20
batch_size = 64
hidden_size = 20
output_size = 3  # Binary classification
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [0,1]

/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_dataset = EEGDataset(train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(test_data_csv, number_of_eeg_channels=number_of_eeg_channels)
combined_dataset = ConcatDataset([train_dataset, val_dataset])

# Split the data into training and validation sets
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)
train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)

val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [3]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = ConvParallelEEG1DModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:57: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at ../aten/src/ATen/native/Copy.cpp:305.)
  coefficients_cmor = torch.tensor(coefficients_cmor, dtype=torch.float32)
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


torch.Size([1, 1, 400]) torch.Size([1, 33, 14]) torch.Size([1, 30, 400])
Output shape: torch.Size([1, 3])


ConvParallelEEG1DModel(
  (branches): ModuleList(
    (0): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(in_features=32, out_features=3, bias=True)
    )
    (1): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(33, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(

In [4]:
prev_eval_loss = 0
prev_eval_acc = 0


In [5]:
model = model.to(device)

prev_eval_loss, prev_eval_acc = eval(model, val_loader, loss_fn, 1,False,'', active_branches, device)
print("Loss:",prev_eval_loss, "Accuracy",prev_eval_acc)

Eval:   0%|          | 0/43343 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x7ca5e669e160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 43343/43343 [03:06<00:00, 232.51it/s]

Correct classified: 5849.0 / 43343
Class-wise correct: Class 0 = 0.0, Class 1 = 0.0, Class 2 = 5849.0
Precision: 4.50  Recall: 33.33  F1: 7.93
Confusion Matrix:
 [[    0     0 22538]
 [    0     0 14956]
 [    0     0  5849]]
Loss: 1.1518479723358612 Accuracy 0.1349468195556376



/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [6]:
result = {"epoch":[],"train_loss":[],"train_acc":[],"eval_loss":[],"eval_acc":[],"test_acc":[],"test_loss":[]}
saved_epoch = 0

In [7]:
for epoch in range(epochs):  # loop over the dataset multiple times
    train_loss, train_acc = train(model, train_loader,optimizer, loss_fn, active_branches, device)
    print(f"Train Loss after epoch {epoch+2}: {train_loss} Train ACC after epoch {epoch}: {train_acc}")
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    print(f"VAL Loss after epoch {epoch}: {eval_loss} VAL ACC after epoch {epoch}: {eval_acc}")
    # test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    test_loss = eval_loss
    test_acc = eval_acc
    print(f"Test Loss after epoch {epoch}: {test_loss} Test Acc after epoch {epoch}: {test_acc}")

    scheduler.step(eval_loss)
    
    result['epoch'].append(epoch)
    
    result['train_loss'].append(train_loss)
    result['train_acc'].append(train_acc)
    result['eval_loss'].append(eval_loss)
    result['eval_acc'].append(eval_acc)
    result['test_loss'].append(test_loss)
    result['test_acc'].append(test_acc)

    if prev_eval_acc < eval_acc:
      prev_eval_acc = eval_acc
      saved_epoch = epoch
      torch.save(model.state_dict(), model_saving_name)
      print("WEIGHTS SAVED")

Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x7ca5e669e160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Training: 100%|██████████| 4531/4531 [11:09<00:00,  6.76it/s]


Correct Classified: 188902.0 / 289972
Class-wise correct: Class 0 = 118856.0, Class 1 = 23504.0, Class 2 = 46542.0
Train Loss after epoch 2: 1.7260208891285778 Train ACC after epoch 0: 0.6514491054308692


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x7ca5e669e160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 74774/74774 [04:26<00:00, 280.82it/s]


Correct classified: 54200.0 / 74774
Class-wise correct: Class 0 = 31557.0, Class 1 = 8154.0, Class 2 = 14489.0
Precision: 68.16  Recall: 68.69  F1: 68.24
Confusion Matrix:
 [[31557  1442  3564]
 [ 1437  8154  3581]
 [ 5410  5140 14489]]
VAL Loss after epoch 0: 0.7670723067209797 VAL ACC after epoch 0: 0.7248508839971113
Test Loss after epoch 0: 0.7670723067209797 Test Acc after epoch 0: 0.7248508839971113
WEIGHTS SAVED


...

Correct classified: 57757.0 / 74774
Class-wise correct: Class 0 = 32138.0, Class 1 = 9374.0, Class 2 = 16245.0
Precision: 72.98  Recall: 74.65  F1: 73.10
Confusion Matrix:
 [[32138  2073  2352]
 [ 1194  9374  2604]
 [ 1917  6877 16245]]
VAL Loss after epoch 18: 0.6801091985903615 VAL ACC after epoch 18: 0.77242089496349
Test Loss after epoch 18: 0.6801091985903615 Test Acc after epoch 18: 0.77242089496349


Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x7ca5e669e160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Training: 100%|██████████| 4531/4531 [13:00<00:00,  5.80it/s]


Correct Classified: 229064.0 / 289972
Class-wise correct: Class 0 = 122854.0, Class 1 = 41668.0, Class 2 = 64542.0
Train Loss after epoch 21: 1.476430105889404 Train ACC after epoch 19: 0.7899521333094229


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x7ca5e669e160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 74774/74774 [04:38<00:00, 268.19it/s]


Correct classified: 57582.0 / 74774
Class-wise correct: Class 0 = 32204.0, Class 1 = 8377.0, Class 2 = 17001.0
Precision: 72.03  Recall: 73.19  F1: 72.33
Confusion Matrix:
 [[32204  1674  2685]
 [ 1558  8377  3237]
 [ 1687  6351 17001]]
VAL Loss after epoch 19: 0.6730931515704761 VAL ACC after epoch 19: 0.7700805092679274
Test Loss after epoch 19: 0.6730931515704761 Test Acc after epoch 19: 0.7700805092679274


In [8]:
print("Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  ")
for epoch in range(epochs):
    if(saved_epoch == epoch):
        print("Saved Weights")
    print(result['epoch'][epoch],"|",result['train_loss'][epoch],"|",result['train_acc'][epoch],"|",result['eval_loss'][epoch],"|",result['eval_acc'][epoch],"|",result['test_loss'][epoch],"|",result['test_acc'][epoch])

Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  
0 | 1.7260208891285778 | 0.6514491054308692 | 0.7670723067209797 | 0.7248508839971113 | 0.7670723067209797 | 0.7248508839971113
1 | 1.5908155940642448 | 0.7269081152663016 | 0.7135739833962287 | 0.7534035894829754 | 0.7135739833962287 | 0.7534035894829754
2 | 1.5565125984851953 | 0.7479998068779055 | 0.6948183821042198 | 0.7721534223125686 | 0.6948183821042198 | 0.7721534223125686
3 | 1.5379551311540067 | 0.75935952436787 | 0.6906158838526826 | 0.7698932784122824 | 0.6906158838526826 | 0.7698932784122824
4 | 1.5248508161594472 | 0.7669947443201413 | 0.6905913419630342 | 0.7680744643860166 | 0.6905913419630342 | 0.7680744643860166
5 | 1.5150250786583672 | 0.7709158125612128 | 0.6722244736536854 | 0.7795356674780004 | 0.6722244736536854 | 0.7795356674780004
6 | 1.5071563393673448 | 0.7746058240105941 | 0.6668107484041087 | 0.7737983791157355 | 0.6668107484041087 | 0.7737983791157355
Saved Weights
7 | 1.5013121

In [9]:
model_state_dict = torch.load(model_saving_name)
# Load the state_dict into the model
model.load_state_dict(model_state_dict)
eval_loss, eval_acc = eval(model, val_loader, loss_fn, eval_batch,True,prediction_saving_csv_eval, active_branches, device)
print(f"VAL Loss: {eval_loss} VAL ACC: {eval_acc}")

# eval_loss, eval_acc = eval(model, train_loader, loss_fn, eval_batch,True,prediction_saving_csv_train)
# print(f"Train Loss: {eval_loss} Train ACC: {eval_acc}")

/tmp/ipykernel_1085736/3846926396.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state_dict = torch.load(model_saving_name)
Eval:   0%|          | 0/43343 [00:00<

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 43343/43343 [02:46<00:00, 261.05it/s]


Correct classified: 34390.0 / 43343
Class-wise correct: Class 0 = 18985.0, Class 1 = 11345.0, Class 2 = 4060.0
Precision: 73.76  Recall: 76.50  F1: 74.25
Confusion Matrix:
 [[18985  1455  2098]
 [ 1047 11345  2564]
 [ 1451   338  4060]]
VAL Loss: 0.6613021071443229 VAL ACC: 0.7934383868214013
